# Warfarin QML — Classical matched-input baselines (PCA k=4)

**Mục đích:** Tái lập, lưu đầy đủ metrics / predictions / confusion / config cho 6 cấu hình Classical trên **Quantum Development Pool 600 mẫu**, **5-fold CV**, mỗi fold **480 train / 120 validation**. Đối chiếu với các kết quả Classical đã khôi phục từ Terminal.

**Dữ liệu:** IWPC đã được lọc và đánh nhãn theo Feature Policy 10 cột. `Final Test` đã khóa, **không được dùng trong notebook này**.

**Nguyên tắc:** fit imputation/encoding/scaling/PCA/MinMax chỉ trên training fold. Cùng PCA k=4 và `[0,π]` clipping cho matched-input Classical và Quantum. Không chỉnh siêu tham số dựa trên validation.

**Trạng thái:** Notebook mới để *tái lập*, chưa phải chứng cứ đã chạy. Kết quả hiện có `classical_unweighted_5fold.csv`, `classical_weighted_5fold.csv`, `classical_reported_summaries.csv` **không được ghi đè**. Sau khi chạy, các file mới mang tiền tố `classical_k4_reproduced_` nằm trong `docs/results/`.

**Môi trường:** VS Code → chọn kernel Conda `nt409` (Python với `pandas`, `PyYAML`, `xlrd`, `numpy`, `scikit-learn >=1.7`). **Chạy lần lượt từ cell đầu**. Đọc các cảnh báo trước khi chạy cell train.

## 1. Imports, project root và kiểm tra môi trường

Notebook có thể đặt trong `notebooks/` hoặc mở từ thư mục gốc project; tự tìm thư mục chứa `configs/feature_policy.yaml`.

In [1]:
from pathlib import Path
import sys, json, hashlib, warnings
from importlib.metadata import version
from time import perf_counter

import numpy as np
import pandas as pd
import yaml
import sklearn
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, recall_score, precision_score,
                             confusion_matrix)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight


def find_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if ((candidate / "configs/feature_policy.yaml").is_file()
                and (candidate / "data/splits/quantum_dev_folds.csv").is_file()):
            return candidate
    raise FileNotFoundError(
        "Open this notebook inside Warfarin project; could not find configs/ and data/splits/"
    )

ROOT = find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.prepare_stage4 import load_and_validate_dataset, verify_raw_file
from src.preprocessing import build_preprocessor

OUT = ROOT / "docs/results"
CLASSES = ["Low", "Intermediate", "High"]
PREFIX = "classical_k4_reproduced"
FILES = {
    "metrics": OUT / f"{PREFIX}_metrics.csv",
    "predictions": OUT / f"{PREFIX}_predictions.csv",
    "confusion": OUT / f"{PREFIX}_confusion.csv",
    "summary": OUT / f"{PREFIX}_summary.csv",
    "config": OUT / f"{PREFIX}_config.json",
}

print("Project:", ROOT)
print("Python:", sys.executable)
print("scikit-learn:", sklearn.__version__)
print("New outputs:", *(p.name for p in FILES.values()), sep="\n - ")

# Weighted MLP sample_weight is supported in recent scikit-learn.
if tuple(int(x) for x in sklearn.__version__.split(".")[:2]) < (1, 7):
    raise RuntimeError("MLP-weighted requires scikit-learn >=1.7; use nt409 environment")

existing = [str(p) for p in FILES.values() if p.exists()]
if existing:
    raise FileExistsError("Safe stop: reproduced outputs already exist; never overwrite. " + str(existing))


Project: D:\Warfarin-Quantum-Machine-Learning-Classification
Python: c:\Users\admin\anaconda3\envs\nt409\python.exe
scikit-learn: 1.9.1
New outputs:
 - classical_k4_reproduced_metrics.csv
 - classical_k4_reproduced_predictions.csv
 - classical_k4_reproduced_confusion.csv
 - classical_k4_reproduced_summary.csv
 - classical_k4_reproduced_config.json


## 2. Khóa dữ liệu/splits và kiểm tra rò rỉ

Đọc lại dữ liệu gốc đúng hàm GĐ4, kiểm tra hash file raw và toàn bộ split. Kiểm tra tất cả 600 sample IDs đều thuộc Development, nằm ngoài Final Test, và 5 folds phân bổ đủ.

In [2]:
def sha256_file(path: Path) -> str:
    hasher = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            hasher.update(block)
    return hasher.hexdigest()

with (ROOT / "configs/feature_policy.yaml").open(encoding="utf-8") as fh:
    policy = yaml.safe_load(fh)
assert policy["training_allowed"] is True
approved = list(policy["approved_X_columns"])
assert len(approved) == len(set(approved)) == 10

verify_raw_file()
split_dir = ROOT / "data/splits"
manifest_path = split_dir / "split_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest["status"] == "LOCKED_BEFORE_MODEL_PERFORMANCE"
assert manifest["approved_features"] == approved
for filename, expected in manifest["files"].items():
    assert sha256_file(split_dir / filename) == expected, f"Checksum mismatch: {filename}"

X, y = load_and_validate_dataset(approved)
labels = pd.read_csv(ROOT / "data/interim/labels.csv")
folds = pd.read_csv(split_dir / "quantum_dev_folds.csv")
splits = pd.read_csv(split_dir / "final_split.csv")
y_array = np.asarray(y, dtype=str)

assert len(X) == len(y_array) == len(labels) == 5410
assert labels["raw_row_index"].is_unique
assert np.array_equal(y_array, labels["target_group"].astype(str).to_numpy())
assert folds["raw_row_index"].is_unique
assert set(folds["fold_id"].astype(int)) == set(range(5))

qdev_ids = set(folds["raw_row_index"].astype(int))
dev_ids = set(splits.loc[splits["partition"] == "development", "raw_row_index"].astype(int))
test_ids = set(splits.loc[splits["partition"] == "final_test", "raw_row_index"].astype(int))
assert len(qdev_ids) == 600
assert qdev_ids.issubset(dev_ids)
assert dev_ids.isdisjoint(test_ids)
assert qdev_ids.isdisjoint(test_ids)
assert dev_ids | test_ids == set(labels["raw_row_index"].astype(int))
assert folds.groupby("fold_id").size().to_dict() == {i: 120 for i in range(5)}

print("[PASS] Raw/checksum and approved feature policy")
print("[PASS] 5410 labeled samples; quantum pool 600; five validation folds of 120")
print("[PASS] Final Test excluded")


[PASS] Raw/checksum and approved feature policy
[PASS] 5410 labeled samples; quantum pool 600; five validation folds of 120
[PASS] Final Test excluded


## 3. Cấu hình cố định — 6 Classical models

Không tuning trong notebook này. `MLP-weighted` dùng `sample_weight` tính **chỉ từ nhãn training fold**; các mô hình còn lại không dùng trọng số mẫu trực tiếp. Các cấu hình chính khớp các thử nghiệm đã báo cáo trong cuộc trò chuyện.

In [3]:
MODELS = ["LR", "SVM", "MLP", "LR-balanced", "SVM-balanced", "MLP-weighted"]

def create_model(name):
    if name == "LR":
        return LogisticRegression(max_iter=2000, random_state=42)
    if name == "SVM":
        return SVC(kernel="rbf", C=1.0, gamma="scale")
    if name == "MLP":
        return MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500,
                             early_stopping=True, n_iter_no_change=20,
                             random_state=42)
    if name == "LR-balanced":
        return LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
    if name == "SVM-balanced":
        return SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced")
    if name == "MLP-weighted":
        return MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500,
                             early_stopping=True, n_iter_no_change=20,
                             random_state=42)
    raise ValueError(f"Unknown model: {name}")

print("[PASS] Models:", MODELS)


[PASS] Models: ['LR', 'SVM', 'MLP', 'LR-balanced', 'SVM-balanced', 'MLP-weighted']


## 4. Huấn luyện + đánh giá 5 folds

**Cell thực sự huấn luyện** — tất cả transforms fit trên training fold. Mỗi model trong mỗi fold dùng một estimator mới. Nếu kết quả cũ đã tồn tại, Cell 1 đã dừng trước khi chạy. Kết quả tạm thời chỉ nằm trong biến của notebook; Cell 5 mới lưu ra đĩa.

In [4]:
metrics_rows, predictions_rows, confusion_rows = [], [], []

for fold_id in range(5):
    val_ids = set(folds.loc[folds["fold_id"] == fold_id, "raw_row_index"].astype(int))
    train_ids = qdev_ids - val_ids
    assert len(train_ids) == 480 and len(val_ids) == 120
    assert train_ids.isdisjoint(val_ids) and (train_ids | val_ids) == qdev_ids
    assert (train_ids | val_ids).isdisjoint(test_ids)

    train_pos = np.flatnonzero(labels["raw_row_index"].isin(train_ids).to_numpy())
    val_pos = np.flatnonzero(labels["raw_row_index"].isin(val_ids).to_numpy())
    X_train, X_val = X.iloc[train_pos], X.iloc[val_pos]
    y_train, y_val = y_array[train_pos], y_array[val_pos]

    prep = build_preprocessor(approved)
    encoded_train = prep.fit_transform(X_train)
    encoded_val = prep.transform(X_val)
    pca = PCA(n_components=4, svd_solver="full")
    reduced_train = pca.fit_transform(encoded_train)
    reduced_val = pca.transform(encoded_val)
    scaler = MinMaxScaler(feature_range=(0.0, float(np.pi)))
    z_train = np.clip(scaler.fit_transform(reduced_train), 0, np.pi)
    z_val = np.clip(scaler.transform(reduced_val), 0, np.pi)
    assert z_train.shape == (480, 4) and z_val.shape == (120, 4)
    assert np.isfinite(z_train).all() and np.isfinite(z_val).all()

    for name in MODELS:
        model = create_model(name)
        tic = perf_counter()
        if name == "MLP-weighted":
            weights = compute_sample_weight(class_weight="balanced", y=y_train)
            model.fit(z_train, y_train, sample_weight=weights)
        else:
            model.fit(z_train, y_train)
        fit_seconds = perf_counter() - tic

        tic = perf_counter()
        predicted = np.asarray(model.predict(z_val), dtype=str)
        predict_seconds = perf_counter() - tic
        assert len(predicted) == 120 and set(predicted).issubset(set(CLASSES))

        recall = recall_score(y_val, predicted, labels=CLASSES, average=None, zero_division=0)
        precision = precision_score(y_val, predicted, labels=CLASSES, average=None, zero_division=0)
        cm = confusion_matrix(y_val, predicted, labels=CLASSES)
        metrics_rows.append({
            "fold": fold_id, "model": name, "k": 4, "train_n": 480, "val_n": 120,
            "macro_f1": f1_score(y_val, predicted, labels=CLASSES, average="macro", zero_division=0),
            "accuracy": accuracy_score(y_val, predicted),
            "balanced_accuracy": balanced_accuracy_score(y_val, predicted),
            "recall_high": float(recall[2]), "precision_high": float(precision[2]),
            "true_high": int(np.sum(y_val == "High")),
            "pred_high": int(np.sum(predicted == "High")),
            "pca_explained_variance": float(pca.explained_variance_ratio_.sum()),
            "fit_seconds": fit_seconds, "predict_seconds": predict_seconds,
        })
        for idx, actual, pred in zip(val_pos, y_val, predicted):
            predictions_rows.append({
                "fold": fold_id, "model": name,
                "raw_row_index": int(labels.iloc[idx]["raw_row_index"]),
                "actual": str(actual), "predicted": str(pred)
            })
        for i, actual in enumerate(CLASSES):
            for j, pred in enumerate(CLASSES):
                confusion_rows.append({
                    "fold": fold_id, "model": name,
                    "actual": actual, "predicted": pred, "count": int(cm[i, j])
                })
        print(f"[PASS] Fold {fold_id} | {name:14s} | F1={metrics_rows[-1]['macro_f1']:.4f} | High recall={recall[2]:.4f}")

metrics_df = pd.DataFrame(metrics_rows)
predictions_df = pd.DataFrame(predictions_rows)
confusion_df = pd.DataFrame(confusion_rows)
assert len(metrics_df) == 30
assert len(predictions_df) == 3600
assert len(confusion_df) == 270
assert predictions_df.groupby("model")["raw_row_index"].nunique().eq(600).all()
assert set(metrics_df["model"]) == set(MODELS)
print("[PASS] 30 model-fold rows / 3600 predictions / 270 confusion cells")


[PASS] Fold 0 | LR             | F1=0.3963 | High recall=0.0000
[PASS] Fold 0 | SVM            | F1=0.4250 | High recall=0.0000
[PASS] Fold 0 | MLP            | F1=0.4338 | High recall=0.0000
[PASS] Fold 0 | LR-balanced    | F1=0.3859 | High recall=0.5333
[PASS] Fold 0 | SVM-balanced   | F1=0.4228 | High recall=0.4667
[PASS] Fold 0 | MLP-weighted   | F1=0.3165 | High recall=0.0000
[PASS] Fold 1 | LR             | F1=0.3565 | High recall=0.0000
[PASS] Fold 1 | SVM            | F1=0.3784 | High recall=0.0000
[PASS] Fold 1 | MLP            | F1=0.3668 | High recall=0.0000
[PASS] Fold 1 | LR-balanced    | F1=0.3841 | High recall=0.5333
[PASS] Fold 1 | SVM-balanced   | F1=0.3916 | High recall=0.4667
[PASS] Fold 1 | MLP-weighted   | F1=0.4523 | High recall=0.4667
[PASS] Fold 2 | LR             | F1=0.3933 | High recall=0.0000
[PASS] Fold 2 | SVM            | F1=0.3863 | High recall=0.0000
[PASS] Fold 2 | MLP            | F1=0.3106 | High recall=0.0000
[PASS] Fold 2 | LR-balanced    | F1=0.38

## 5. Tổng hợp, đối chiếu số liệu đã khôi phục, lưu riêng để bảo vệ kết quả cũ

Lưu metrics full precision, mỗi sample prediction, confusion matrix và đầy đủ config. Đối chiếu chênh lệch với bản khôi phục từ Terminal (bản cũ có thể làm tròn). Không tự động ghi đè file nào.

In [5]:
summary_df = (
    metrics_df.groupby("model", sort=False)
    .agg(
        folds=("fold", "count"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_sd=("macro_f1", "std"),
        accuracy_mean=("accuracy", "mean"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        recall_high_mean=("recall_high", "mean"),
        fit_seconds_mean=("fit_seconds", "mean"),
    ).reset_index().sort_values("macro_f1_mean", ascending=False)
)
assert len(summary_df) == 6 and summary_df["folds"].eq(5).all()

recovered_path = OUT / "classical_reported_summaries.csv"
if recovered_path.exists():
    old = pd.read_csv(recovered_path)[["model", "macro_f1_mean", "accuracy_mean"]]
    compared = summary_df.merge(old, on="model", suffixes=("_reproduced", "_recovered"), validate="one_to_one")
    compared["macro_f1_delta"] = compared["macro_f1_mean_reproduced"] - compared["macro_f1_mean_recovered"]
    compared["accuracy_delta"] = compared["accuracy_mean_reproduced"] - compared["accuracy_mean_recovered"]
    print("=== Comparison with recovered (rounded) results ===")
    display(compared.round(6))
    if compared["macro_f1_delta"].abs().max() > 0.002:
        print("[REVIEW] Macro-F1 differs >0.002; investigate versions/configuration before handover")

print("=== Reproduced Classical 5-fold Summary ===")
display(summary_df.round(5))

config = {
    "experiment": "Classical_matched_input_PCA_k4_5fold",
    "status": "REPRODUCED_FROM_NOTEBOOK",
    "models": MODELS,
    "data_pool": "quantum_dev_pool_600",
    "cv_folds": 5,
    "features": approved,
    "pca_k": 4,
    "preprocessing": "train_fold_fit_only (src.preprocessing.build_preprocessor)",
    "quantum_matched_scaling": "train_fit_MinMax_0_pi_then_clip",
    "models_configuration": {
        "LR": "LogisticRegression(max_iter=2000, random_state=42)",
        "LR-balanced": "LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced')",
        "SVM": "SVC(kernel='rbf', C=1.0, gamma='scale')",
        "SVM-balanced": "SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced')",
        "MLP": "MLPClassifier(hidden_layer_sizes=(32,16), max_iter=500, early_stopping=True, n_iter_no_change=20, random_state=42)",
        "MLP-weighted": "MLP same as unweighted, fit(sample_weight=compute_sample_weight('balanced',y_train))"
    },
    "class_order": CLASSES,
    "random_state": 42,
    "final_test_used": False,
    "versions": {name: version(name) for name in ("scikit-learn", "pandas", "numpy", "PyYAML")},
    "sha256": {
        "feature_policy": sha256_file(ROOT / "configs/feature_policy.yaml"),
        "split_manifest": sha256_file(manifest_path),
        "raw_workbook": sha256_file(ROOT / "data/raw/PS206767-553247439.xls"),
    },
    "note": "Reproduced experiment; original recovered results remain unchanged."
}

# Last safety gate before any output is created.
existing = [str(p) for p in FILES.values() if p.exists()]
if existing:
    raise FileExistsError("Refusing to overwrite reproduced outputs: " + str(existing))

OUT.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(FILES["metrics"], index=False)
predictions_df.to_csv(FILES["predictions"], index=False)
confusion_df.to_csv(FILES["confusion"], index=False)
summary_df.to_csv(FILES["summary"], index=False)
with FILES["config"].open("x", encoding="utf-8") as handle:
    json.dump(config, handle, ensure_ascii=False, indent=2, allow_nan=False)

print("[PASS] Saved 5 reproduced Classical artifacts (no original files overwritten)")
for name, path in FILES.items():
    print(name, "=>", path)


=== Comparison with recovered (rounded) results ===


,model,folds,macro_f1_mean_reproduced,macro_f1_sd,accuracy_mean_reproduced,balanced_accuracy_mean,recall_high_mean,fit_seconds_mean,macro_f1_mean_recovered,accuracy_mean_recovered,macro_f1_delta,accuracy_delta
0,SVM-balanced,5,0.418669,0.032822,0.440000,0.467382,0.475000,0.006115,0.4187,0.4400,-0.000031,0.000000
1,SVM,5,0.401360,0.022815,0.613333,0.426546,0.000000,0.006604,0.4014,0.6133,-0.000040,0.000033
2,LR-balanced,5,0.391748,0.043564,0.411667,0.461727,0.489167,0.009642,0.3917,0.4117,0.000048,-0.000033
3,LR,5,0.384130,0.017811,0.586667,0.410210,0.000000,0.010114,0.3841,0.5867,0.000030,-0.000033
4,MLP-weighted,5,0.351613,0.095064,0.411667,0.379242,0.225000,0.106325,0.3516,0.4117,0.000013,-0.000033
5,MLP,5,0.342306,0.060184,0.580000,0.384681,0.000000,0.075300,0.3423,0.5800,0.000006,0.000000


=== Reproduced Classical 5-fold Summary ===


,model,folds,macro_f1_mean,macro_f1_sd,accuracy_mean,balanced_accuracy_mean,recall_high_mean,fit_seconds_mean
4,SVM-balanced,5,0.41867,0.03282,0.44000,0.46738,0.47500,0.00611
1,SVM,5,0.40136,0.02281,0.61333,0.42655,0.00000,0.00660
3,LR-balanced,5,0.39175,0.04356,0.41167,0.46173,0.48917,0.00964
0,LR,5,0.38413,0.01781,0.58667,0.41021,0.00000,0.01011
5,MLP-weighted,5,0.35161,0.09506,0.41167,0.37924,0.22500,0.10632
2,MLP,5,0.34231,0.06018,0.58000,0.38468,0.00000,0.07530


[PASS] Saved 5 reproduced Classical artifacts (no original files overwritten)
metrics => D:\Warfarin-Quantum-Machine-Learning-Classification\docs\results\classical_k4_reproduced_metrics.csv
predictions => D:\Warfarin-Quantum-Machine-Learning-Classification\docs\results\classical_k4_reproduced_predictions.csv
confusion => D:\Warfarin-Quantum-Machine-Learning-Classification\docs\results\classical_k4_reproduced_confusion.csv
summary => D:\Warfarin-Quantum-Machine-Learning-Classification\docs\results\classical_k4_reproduced_summary.csv
config => D:\Warfarin-Quantum-Machine-Learning-Classification\docs\results\classical_k4_reproduced_config.json


## 6. Diễn giải và bước bàn giao

- Thí nghiệm **matched benchmark**: 6 cấu hình Classical dùng cùng 600 mẫu và cùng folds với QSVC/VQC; không được trộn với **Strong Classical Reference** trên toàn Development Set (4.328 mẫu), chưa chạy trong notebook này.
- `classical_k4_reproduced_*` là kết quả mới từ notebook; giữ các file `classical_*_5fold.csv` phục hồi để truy vết lịch sử. So sánh sai khác trước khi thay thế trong báo cáo.
- Không dựa vào hiệu năng Final Test vì notebook này chưa và không được dùng Final Test.
- `MLP-weighted` có tính ngẫu nhiên do early stopping; sai khác nhỏ so với bản trước có thể xảy ra tùy version sklearn.
- Nếu có khác biệt lớn, giữ nguyên cả hai, không sửa tay metric. Ghi nguyên nhân từ config/version.
- Không sử dụng kết quả để khuyến nghị liều warfarin lâm sàng: **temporal evidence UNVERIFIED** theo Feature Policy.
- Notebook là code tái lập cần **giữ**. Khi làm QSVC/VQC notebooks thay thế, kiểm tra tương đương rồi mới xóa những script cũ bị trùng.